# Classification: Supervised Machine Learning
This notebook contains the code to classify the articles within each dataset using supervised machine learning models.

## Custom Modules
The ``src`` directory houses custom modules with functions that will be reused throughout the project.

To be able to import these moduls, we begin with programmatically adding the project's root directory to ``sys.path``.

After adding the root to ``sys.path``, we can import the ``data`` and ``util`` modules:

In [1]:
import os, sys

# recursively search for the root directory containing a specific file
def find_root_dir(search_for='.gitignore'):

    current_dir = os.getcwd()

    while True:
        if os.path.exists(os.path.join(current_dir, search_for)):
            return current_dir
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{search_for}' in any parent directory.")
        current_dir = parent_dir


# save the root directory to a variable
root_dir = find_root_dir()
print(f"Root directory found: {root_dir}")

# add the root directory to the system path
sys.path.append(root_dir)
if root_dir in sys.path:
    print(f"Root directory added to system path.")


# import custom modules
from src import data

Root directory found: c:\dev\automated_title_abstract_screening
Root directory added to system path.


## Loading the Data Dictionary

In [2]:
# where the data is stored
data_directory = '../../data/datasets/04_preprocessed/supervised'

# load the data
datasets = data.dict_from_directory(data_directory, type='polars')

## Helper Functions
We define two helper functions to create train-test-datasets and to construct a pipeline with tf-idf vectorization and random undersampling:

In [3]:
import polars as pl
from sklearn.model_selection import train_test_split

def create_train_test_sets(df):
    """
    Return a train-test split of the data

    Args:
        df: Polars DataFrame with columns 'title', 'abstract', and 'include'.

    Returns:
        X_train: list of strings, training data.
        X_test: list of strings, test data.
        y_train: list of booleans, training labels.
        y_test: list of booleans, test labels.
    """
    # combine title and abstract into one column
    # fill null values with empty strings to avoid errors
    combined = df.select(
        pl.col('index'),
        pl.concat_str(
            pl.col('title').fill_null(''),
            pl.col('abstract').fill_null(''),
            separator=' ',
        ).alias('text')
    )

    # features
    #X = combined.to_series().to_numpy()
    X = combined.to_pandas()
    X.set_index('index', inplace=True)

    # target
    #y = df['include'].to_numpy()
    y = df['include'].to_pandas()
    y = y.set_axis(X.index)


    # train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, 
        y, 
        test_size=0.3, # the test set will be 30% of the data
        random_state=42,  # important for reproducibility
        stratify=y # important for imbalanced classes
    ) 

    # get the indices of the test set
    test_indices = X_test.index

    return X_train, X_test, y_train, y_test, test_indices

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

def create_pipeline(estimator):
    """
    Return a pipeline ready encapuslating a given estimator.
    The pipeline includes Tf-idf vectorization and random undersampling.

    Args:
        estimator: scikit-learn estimator.
    
    Returns:
        pipeline: imbalanced-learn pipeline.
    """
    return Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('undersampling', RandomUnderSampler(
            sampling_strategy='auto',
            random_state=42
            )
        ),
        ('estimator', estimator)
    ])

## Estimators
We will use four established estimators to predict the eligibility of articles.

Note that ``class_weight`` balanced is set for all estimators except for ComplementNB, which does not support it to account for class imbalance.

Other than that, the arguments remain at default values

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import ComplementNB

# dictionary of estimators to predict with
# class weights will be used for estimators which support it
estimators = {
    'logistic_regression': LogisticRegression(class_weight='balanced'),
    'random_forest': RandomForestClassifier(class_weight='balanced'),
    'support_vector_machine': SVC(class_weight='balanced'),
    'naive_bayes': ComplementNB(),
}

## Prediction
Below this headline, we create train-test-datasets for each dataset, which we use to fit the estimators on.
We then predict the eligibility of articles by these fitted estimators.

We save the predictions within a dictionary where the keys refer to the datasets and the values contain pl.DataFrames with the predictions.

The ``predictions`` dataframe contains the ground truth, thus the decision by human reviewers within the ``true`` column, and the decision by each estimator in respective columns next to it. 

With this format we evaluate the model performance within the next notebooks and calculate performance metrics.

In [6]:
from tqdm.notebook import tqdm

# dictionary to store predictions for each dataset and estimator
predictions = {}

# iterate over datasets
for subject, dataset in tqdm(
    iterable=datasets.items(),
    desc='Datasets',
    total=len(datasets),
    leave=True
):

    # use the same train-test split for all estimators
    X_train, X_test, y_train, y_test, test_indices = create_train_test_sets(
        dataset
    )

    # dataframe to store true values and predictions per estimator
    predictions[subject] = pl.DataFrame(
        {
            'index': test_indices,
            'true': y_test
        }
    )

    # convert to numpy arrays
    X_train = X_train['text'].to_numpy()
    y_train = y_train.to_numpy()
    X_test = X_test['text'].to_numpy()

    # iterate over estimators
    for name, estimator in tqdm(
        iterable=estimators.items(),
        desc='Estimators',
        total=len(estimators),
        leave=False,
    ):
        
        # create a prediction pipeline
        pipeline = create_pipeline(estimator)

        # fit the estimator
        pipeline.fit(X_train, y_train)

        # predict class labels
        y_pred = pipeline.predict(X_test)

        # store the predictions
        predictions[subject] = predictions[subject].with_columns(
            pl.Series(name=name, values=y_pred)
        )

Datasets:   0%|          | 0/5 [00:00<?, ?it/s]

Estimators:   0%|          | 0/4 [00:00<?, ?it/s]

Estimators:   0%|          | 0/4 [00:00<?, ?it/s]

Estimators:   0%|          | 0/4 [00:00<?, ?it/s]

Estimators:   0%|          | 0/4 [00:00<?, ?it/s]

Estimators:   0%|          | 0/4 [00:00<?, ?it/s]

## Export

In [ ]:
# where to save the predictions
export_path = '../../data/predictions/supervised_machine_learning'

for subject, df in predictions.items():
    df.write_csv(f'{export_path}/{subject}_pred.csv')